In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In this Jupyter Notebook we compare performances of standard the variable inclusion method used for variable selection with a modified log-likelihood method.

### Standard Variable Inclusion
The BART algorithm produces a large number of tree ensambles (posterior samples), each containing _m_ trees. For each posterior sample we calculate the proportion of times each variable was used in a splitting rule among all splitting variables. Averaging the proportions across all posterior samples we estimate the _variable inclusion proportion_ of each variable.

An example for ensamble with $m=3$ trees can be seen in the figure below. Once we calculate inclusion proportions $v_k^{(s)}$ for all $S$ posterior samples, we get the estimated _variable inclusion proportions_ by averaging $p_k = \frac{1}{S}\sum_s^Sv_k^{(s)}$ for all $K$ variables. 

<div
  role="img"
  aria-label="Variable inclusion proportions in BART"
  style="
    width: min(900px, 100%);
    height: 340px;
    margin: 1rem auto;
    background-color: currentColor;
    -webkit-mask: url('figures/bart_vi_standard.svg') center / contain no-repeat;
    mask: url('figures/bart_vi_standard.svg') center / contain no-repeat;
  "
></div>

Intuitively, a large variable inclusion proportion suggests importance of a given variable. This is especially clear when the number of trees in each sample (_m_) is small, forcing the algorithm to be selective with variables.

### Log-likelihood Variable Inclusion
The modified variable inclusion proportion calculates the estimated inclusion proportions in the same way as before, but each appearance of a variable in a splitting rule is weighted with a log-likelihood difference. The idea is that original estimated variable inclusions don't take into account how much the variables improve the fit of the algorithm, while the weights do.

For each internal node a log-likelihood difference is calculated between the current tree and a tree where the node in question is a leaf. Thus the weights represent how much the log-likelihood of a tree is imporved compared to if the split in question (and all subsequent splits) didn't happen. 

<div
  role="img"
  aria-label="Variable inclusion proportions in BART"
  style="
    height: 600px;
    margin: 1rem auto;
    background-color: currentColor;
    -webkit-mask: url('figures/bart_vi_weighted.svg') center / contain no-repeat;
    mask: url('figures/bart_vi_weighted.svg') center / contain no-repeat;
  "
></div>


In [4]:
columns = [
    "scenario", "n", "p", "s2",
    "precision_raw_mean", "precision_logl_mean", "precision_diff",
    "recall_raw_mean", "recall_logl_mean", "recall_diff",
    "f1_raw_mean", "f1_logl_mean", "f1_diff",
]

diff_columns = [
    "precision_diff", "recall_diff", "f1_diff",
]


def show_results(results: pd.DataFrame):
    df = results.copy()

    df["precision_diff"] = (
        df["precision_logl_mean"] - df["precision_raw_mean"]
    )
    df["recall_diff"] = (
        df["recall_logl_mean"] - df["recall_raw_mean"]
    )
    df["f1_diff"] = (
        df["f1_logl_mean"] - df["f1_raw_mean"]
    )

    df = df[columns].round(3)

    def highlight_diff(value):
        if value > 0:
            return "background-color: #6aa876; color: #1f1f1f;"
        if value < 0:
            return "background-color: #c74040; color: #1f1f1f;"
        return ""

    display(
        df.style
        .format(precision=3)
        .map(highlight_diff, subset=diff_columns)
    )

In all simulations the standard additive model is assumed, ie. for a given input $x=(x_1,\dots, x_p)$ the response $Y$ is assumed to be:
$$
    Y = f(x) + \epsilon, \quad \epsilon \sim N(0, \sigma^2)
$$

Bellow are results of standard and weighted variable inclusion proportions for some simulated sets of inputs $x$ and functions $f$. For each set of inputs and outputs, a number of hyperparameter values were tested. Hyperparameters include size of sample (_n_), number of covariates (_p_) and variance of noise ($\sigma^2$). For detection thresholds (how high an inclusion proportion has to be in order to say a variable is "important") the method proposed in _Variable selection for BART: An application to gene regulation_ is used.

The reported scores (_precision_, _recall_, _f1_) are averaged over 10 repeats for each combination of hyperparameters.

The values in difference columns are marked green or red, depending on if the weighted or standard proportions performed better.

In [7]:
cc0_results = pd.read_csv("Experiments/results/cc0.csv")
cc0_results

,scenario,model_key,n,p,s2,repeats_requested,repeats_successful,precision_raw_mean,precision_raw_std,precision_raw_se,...,precision_logl_std,precision_logl_se,recall_logl_mean,recall_logl_std,recall_logl_se,f1_logl_mean,f1_logl_std,f1_logl_se,status,error
0,cc0,continuous,100,50,1.0,10,10,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,complete,NaN


## Continuous Predictors and Continuous Response

In both of these examples $n = 500$ was considered, with $p \in \{50, 200\}$ and $\sigma^2 = 1$.

In the first scenario the Friedman function was used, defined as:
$$
    f(x) = 10 \sin{(\pi x_1 x_2)} + 20(x_3 - 0.5)^2 + 10x_4 + 5x_5,
$$
with $x_1, \dots, x_p \sim Unif(0, 1)$ independent.

In [ ]:
cc1_results = pd.read_csv("results/cc1.csv")
show_results(cc1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cc1,500,50,1.000,0.963,1.000,0.037,0.860,0.800,-0.060,0.901,0.883,-0.018
1,cc1,500,200,1.000,0.397,0.682,0.285,0.600,0.940,0.340,0.476,0.783,0.307


For the second continuous-continuous scenario, detection for correlated covariates was compared.

Here $x_1,\dots,x_p$ came from a multivariate normal distribution $N(0, \Sigma)$ where $\Sigma_ij = 0.3^{|i - j|}$ and $f$ is defined as:
$$
    f(x) = 2 x_1 x_4 + 2 x_7 x_{10}

$$

In [6]:
cc2_results = pd.read_csv("results/cc2.csv")
show_results(cc2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cc2,500,50,1.000,0.192,0.257,0.065,0.125,0.225,0.100,0.149,0.235,0.086
1,cc2,500,200,1.000,0.064,0.076,0.011,0.125,0.225,0.100,0.084,0.112,0.028


## Mixed Predictors and Continuous Response

In these simulations some covariates come from a Bernoulli discrete distribution.

In the first scenario, predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. Function $f$ is:
$$
    f(x) = 10 \sin{(\pi x_{\lceil p/2 \rceil +1} x_{\lceil p/2 \rceil +2})} + 20(x_{\lceil p/2 \rceil +3} - 0.5)^2 + 10x_1 + 5x_2
$$

Here we consider $n \in \{ 500, 1000 \}$, $p \in \{ 50, 200 \}$ and $\sigma^2 \in \{1, 10\}$.

In [7]:
cm1_results = pd.read_csv("results/cm1.csv")
show_results(cm1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cm1,500,50,1.000,0.930,1.000,0.070,0.580,0.780,0.200,0.703,0.872,0.170
1,cm1,500,50,10.000,0.892,1.000,0.108,0.460,0.780,0.320,0.597,0.875,0.278
2,cm1,500,200,1.000,0.287,0.532,0.244,0.440,0.820,0.380,0.331,0.630,0.299
3,cm1,500,200,10.000,0.318,0.500,0.182,0.480,0.860,0.380,0.379,0.624,0.245
4,cm1,1000,50,1.000,0.930,1.000,0.070,0.640,0.740,0.100,0.752,0.844,0.092
5,cm1,1000,50,10.000,0.950,1.000,0.050,0.580,0.820,0.240,0.715,0.894,0.179
6,cm1,1000,200,1.000,0.342,0.581,0.239,0.580,0.940,0.360,0.424,0.709,0.286
7,cm1,1000,200,10.000,0.318,0.619,0.300,0.500,0.860,0.360,0.384,0.691,0.307


In this scenario, predictors $x_1, \dots, x_{20} \sim Bernoulli(0.5)$, $x_{21}, \dots, x_{40} \sim Bernoulli(0.5)$ and $x_41, \dots, x_{84} \sim Bernoulli(0.5)$ independently, while $f$ is:
$$
    f(x) = -4 + x_1 + \sin(\pi x_1 x_{44}) - x_{21} + 0.6 x_{41} x_{42} - \exp[-2(x_{42} + 1)^2] - x_{43}^2 + 0.5x_{44}
$$

Here we consider $n \in \{500, 1000\}$ and $\sigma^2 \in \{1, 10\}$.

In [8]:
cm2_results = pd.read_csv("results/cm2.csv")
show_results(cm2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,cm2,500,50,1.000,0.606,0.915,0.309,0.433,0.717,0.283,0.493,0.794,0.301
1,cm2,500,50,10.000,0.395,0.697,0.301,0.267,0.550,0.283,0.311,0.611,0.300
2,cm2,1000,50,1.000,0.657,0.983,0.327,0.517,0.767,0.250,0.572,0.858,0.286
3,cm2,1000,50,10.000,0.515,0.902,0.386,0.333,0.633,0.300,0.394,0.739,0.344


### Mixed Predictors and Binary Response

In the first scenario with a binary response, we sample predictors $x_1, \dots, x_{\lceil p/2 \rceil} \sim Bernoulli(0.5)$ independently and $x_{\lceil p/2 \rceil + 1}, \dots, x_p \sim Unif(0, 1)$ independently. We sample the reponse $y$ from $Bernoulli(\Phi(f(x)))$ where $f$ is the same as in the first mixed-continuous scenario.

We try combinations of $n \in \{500, 1000\}$ and $p \in \{50, 200\}$.

In [9]:
bm1_results = pd.read_csv("results/bm1.csv")
show_results(bm1_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,bm1,500,50,1.000,0.053,0.200,0.147,0.040,0.080,0.040,0.045,0.114,0.069
1,bm1,500,200,1.000,0.014,0.050,0.036,0.020,0.100,0.080,0.017,0.066,0.049
2,bm1,1000,50,1.000,0.283,0.083,-0.200,0.100,0.040,-0.060,0.144,0.054,-0.090
3,bm1,1000,200,1.000,0.053,0.020,-0.033,0.060,0.040,-0.020,0.056,0.027,-0.030


In this scenario, we sample predictors $x_1, \dots, x_{20} \sim Bernoulli(0.5)$, $x_{21}, \dots, x_{40} \sim Bernoulli(0.5)$ and $x_{41}, \dots, x_{84} \sim Bernoulli(0.5)$ independently and response $y$ from  $Bernoulli(\Phi(f(x)))$, where $f$ is the same as in the second mixed-continuous scenario.

Here we test $n \in \{500, 1000\}$.

In [10]:
bm2_results = pd.read_csv("results/bm2.csv")
show_results(bm2_results)

,scenario,n,p,s2,precision_raw_mean,precision_logl_mean,precision_diff,recall_raw_mean,recall_logl_mean,recall_diff,f1_raw_mean,f1_logl_mean,f1_diff
0,bm2,500,50,1.000,0.025,0.095,0.070,0.017,0.050,0.033,0.020,0.063,0.043
1,bm2,1000,50,1.000,0.083,0.106,0.022,0.033,0.083,0.050,0.047,0.088,0.041
